# 04 · Evaluate

Scores every finished run's selected checkpoint with the fixed-target metric on **validation**. The test split is scored only once, after every run and decision is frozen.

In [ ]:
%run 02_model.ipynb

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

runs = [json.loads(path.read_text()) for path in sorted(RUNS_DIR.glob("*.json"))]

rows = []
for run in runs:
    config = Config(**run["config"])
    model = CharTransformer(config).to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINT_DIR / config.run_name / "best.pt", map_location=DEVICE))
    rows.append({
        "model": "transformer",
        "context_len": config.context_len,
        "seed": config.seed,
        "best_step": run["best_step"],
        "train_minutes": round(run["train_seconds"] / 60, 1),
        "val_bpc": transformer_bpc(model, val_data, config.context_len),
    })

best_ngram = json.loads((RESULTS_DIR / "ngram.json").read_text())["best"]
rows.append({"model": f"{best_ngram['order']}-gram (d={best_ngram['discount']})", "val_bpc": best_ngram["val_bpc"]})

results = pd.DataFrame(rows)
results.to_csv(RESULTS_DIR / "validation_results.csv", index=False)
results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for run in runs:
    steps = [point["step"] for point in run["history"]]
    ax.plot(steps, [point["train_loss"] for point in run["history"]], label=f"{run['config']['context_len']} train")
    ax.plot(steps, [point["val_loss"] for point in run["history"]], "--", label=f"{run['config']['context_len']} validation")
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy (nats per character)")
ax.set_title("Training curves by context length")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150)